# Arrakis 산림청 특화 모델: Fire & Smoke Detection

산불/연기 탐지를 위한 YOLO26s 파인튜닝

- **Base model**: yolo26s.pt (Arrakis-Project)
- **Dataset**: [D-Fire / Smoke-Fire-Detection-YOLO](https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo) (21,000+ images)
- **Classes**: smoke (0), fire (1)
- **GPU**: P100

### Repository defaults:
- epochs=50
- imgsz=640
- batch=16
- workers=4
- device=0

In [ ]:
import os
from pathlib import Path

input_root = Path('/kaggle/input')
print('Top-level Kaggle inputs:')
for path in sorted(input_root.iterdir()):
    print('-', path)

print('\nFull directory structure (dirs only, with file counts):')
for root, dirs, files in os.walk(input_root):
    rel = os.path.relpath(root, input_root)
    depth = 0 if rel == '.' else rel.count(os.sep) + 1
    indent = ' ' * depth
    dirname = Path(root).name if rel != '.' else 'input'
    txt_count = sum(1 for f in files if f.endswith('.txt'))
    img_count = sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    other = len(files) - txt_count - img_count
    parts = []
    if img_count: parts.append(f'{img_count} images')
    if txt_count: parts.append(f'{txt_count} txt')
    if other: parts.append(f'{other} other')
    suffix = f'  [{', '.join(parts)}]' if parts else ''
    print(f'{indent}{dirname}/{suffix}')

In [ ]:
%%bash
# Clone Arrakis-Project repo and install dependencies
cd /kaggle/working
git clone https://github.com/chadol0130-arch/Arrakis-Project.git
pip install -q ultralytics>=8.4.21

# Verify environment
python -c "
import torch
from ultralytics import __version__ as uv
print(f'Repository ready at: /kaggle/working/Arrakis-Project')
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA capability: {torch.cuda.get_device_capability(0)}')
print(f'ultralytics version: {uv}')
"

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_dir = Path('/kaggle/working/Arrakis-Project')
train_script = repo_dir / 'kaggle_train_fire_smoke_yolo26s.py'

cmd = [sys.executable, str(train_script)]

print(f'Running: {" ".join(cmd)}')
print('=' * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

rc = process.wait()
print('=' * 60)
print(f'Exit code: {rc}')
if rc != 0:
    raise RuntimeError(f'Training script failed with exit code {rc}')